## is it arthritis?

In [1]:
 !pip install -Uqq fastai

In [2]:
import warnings
import logging
warnings.filterwarnings('ignore')
logging.disable(logging.CRITICAL)
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

In [3]:
# It's a good idea to ensure you're running the latest version of any libraries you need.
# `!pip install -Uqq <libraries>` upgrades to the latest version of <libraries>
# NB: You can safely ignore any warnings or errors pip spits out about running as root or incompatibilities
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')

if iskaggle:
    !pip install -Uqq fastai

The basic steps we'll take are:

1. Use a curated Kaggle dataset of knee X-ray images
1. Prepare normal vs arthritis image classes
1. Fine-tune a pretrained neural network to classify them
1. Test the model on unseen X-rays and evaluate accuracy

## Step 1: Load knee X-ray images from dataset

We're using a pre-made knee X-ray dataset from Kaggle instead of searching the web. The dataset has images organized into folders 0-4 based on arthritis severity, where 0 is normal and 4 is severe arthritis.
Let's first explore the dataset structure to find the correct folder path

In [4]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/shashwatwork
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test/2
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test/0
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test/3
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test/1
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/auto_test/4
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/val
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/val/2
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/val/0
/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-datas

Now let's point to the training folder and see what's inside:

In [5]:
from fastai.vision.all import *
path = Path('/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/train')
path.ls()

[Path('/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/train/2'), Path('/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/train/0'), Path('/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/train/3'), Path('/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/train/1'), Path('/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/train/4')]

We'll use only folders 0 (normal) and 4 (severe arthritis) to keep it a simple binary classification problem. Let's copy these into a clean working directory:

## Step 2: Train our model

We'll copy only the normal (0) and severe arthritis (4) folders into a clean working directory for binary classification

In [6]:
from pathlib import Path
import shutil

# Create a new working directory with just 2 classes
new_path = Path('knee_data')
for label, folder in [('normal', '0'), ('arthritis', '4')]:
    dest = new_path/label
    dest.mkdir(exist_ok=True, parents=True)
    src = path/folder
    for f in src.ls():
        shutil.copy(f, dest/f.name)

new_path.ls()

[Path('knee_data/arthritis'), Path('knee_data/normal')]

Let's remove any corrupted images that might cause training to fail:

In [7]:
failed = verify_images(get_image_files(new_path))
failed.map(Path.unlink)
len(failed)

0

To train a model, we'll need `DataLoaders`, which is an object that contains a *training set* (the images used to create a model) and a *validation set* (the images used to check the accuracy of a model -- not used during training). In `fastai` we can create that easily using a `DataBlock`, and view sample images from it:

In [8]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock), 
    get_items=get_image_files, 
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(new_path, bs=32)

dls.show_batch(max_n=6)

Could not do one pass in your dataloader, there is something wrong in it. Please see the stack trace below:


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


Here what each of the `DataBlock` parameters means:

    blocks=(ImageBlock, CategoryBlock),

inputs are images, outputs are categories (normal or arthritis)

    get_items=get_image_files, 

 finds all image files in the path

    splitter=RandomSplitter(valid_pct=0.2, seed=42),

20% of data goes to validation set, 80% for training

    get_y=parent_label,

the folder name (normal/arthritis) is the label

    item_tfms=[Resize(192, method='squish')]

resize all images to 192x192.

Now let's fine-tune a pretrained resnet34 model on our dataset:

In [ ]:
from fastai.callback.progress import ProgressCallback
learn = vision_learner(dls, resnet34, metrics=error_rate)
learn.remove_cb(ProgressCallback)
learn.fine_tune(4)

Fine-tuning means starting with a model already trained on millions of images and adapting it to our specific task — detecting knee arthritis.

## Step 3: Evaluate our model

Let's test our model on a single image first:

In [ ]:
test_img = get_image_files(new_path/'arthritis')[0]
is_arthritis,_,probs = learn.predict(PILImage.create(test_img))
print(f"This is a: {is_arthritis}.")
print(f"Probabilities: {probs}")
print(f"Classes: {dls.vocab}")

Now let's evaluate properly on the unseen test set — images the model has never seen during training:

In [ ]:
base = Path('/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity/test')
test_files = get_image_files(base/'0') + get_image_files(base/'4')
test_dl = dls.test_dl(test_files)
preds, _ = learn.get_preds(dl=test_dl)

# Get predicted classes
pred_classes = preds.argmax(dim=1)

# Create actual labels (0=arthritis, 1=normal based on our vocab)
actual = ([0] * len(get_image_files(base/'4'))) + ([1] * len(get_image_files(base/'0')))

# Calculate accuracy
correct = (pred_classes == tensor(actual)).float().mean()
print(f"Test accuracy: {correct:.4f}")

This gives us a more honest accuracy than the validation score, since these images were completely unseen.
As you can see, in just a few lines of code we built a medical image classifier that can detect knee arthritis from X-rays with reasonable accuracy — something that would have seemed incredibly difficult just a few years ago!
Deep learning is making it easier than ever to build powerful AI applications. The key is understanding the fundamentals — data, training, and proper evaluation.